평균 제곱 오차(Mean Squared Error, MSE) 손실 함수$$\mathcal{L}(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$여기서 $\hat{y}^{(i)} = wx^{(i)} + b$라고 가정할 때, 가중치($w$)와 편향($b$)에 대해 편미분하는 과정을 Python의 sympy 라이브러리를 사용하여 구현

| 구분 | 변수 수 | 모델 | 파라미터 |
|------|--------|------|----------|
| 단순 회귀 | 1개 | $\hat{y}=w_1x_1+b$ | $w$: 1개, $b$: 1개 |
| 다중 회귀   | $n$개            | $\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$ | $w$: $n$개, $b$: 1개 |
|  |  | $\hat{y}=\sum_{i=1}^{n}w_ix_i+b$ <br> = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$| $w$:n, $b$:1 |

In [1]:
import sympy

# 변수 및 기호 정의
w, b, x, y, m = sympy.symbols('w b x y m')
i = sympy.symbols('i', integer=True)

# 개별 데이터 포인트에 대한 예측값 정의: y_hat = w*x + b
y_hat = w * x + b

# 손실 함수 정의 (단일 데이터 포인트 i에 대한 오차의 제곱 형태)
# 전체 합산(Summation)은 미분 후에도 선형성을 유지하므로,
# 프로그래밍 시에는 대표 식을 미분한 후 합산 기호를 붙이는 방식으로 이해하면 쉽습니다.
loss_individual = (y_hat - y)**2
loss_function = (1 / (2 * m)) * loss_individual

# 1. w(가중치)에 대한 편미분
grad_w = sympy.diff(loss_function, w)

# 2. b(편향)에 대한 편미분
grad_b = sympy.diff(loss_function, b)

print("--- 미분 결과 ---")
print(f"dL/dw: {grad_w}")
print(f"dL/db: {grad_b}")

# 보기 좋게 정리 (Simplify)
print("\n--- 정리된 결과 ---")
print(f"dL/dw = {sympy.simplify(grad_w)}")
print(f"dL/db = {sympy.simplify(grad_b)}")

--- 미분 결과 ---
dL/dw: x*(b + w*x - y)/m
dL/db: (2*b + 2*w*x - 2*y)/(2*m)

--- 정리된 결과 ---
dL/dw = x*(b + w*x - y)/m
dL/db = (b + w*x - y)/m


## 1. Sympy로 대수학과 미분방정식으로 해를 구하기

In [2]:
import sympy
import numpy as np
from sklearn.datasets import make_regression

def solve_regression_symbolic():
    # ---------------------------------------------------------
    # 1. 단순 선형 회귀 (Simple Linear Regression)
    # ---------------------------------------------------------
    print("--- [1] 단순 선형 회귀 (Symbolic) ---")

    # 실제 데이터 생성 (m=100 샘플, 특성 1개)
    m = 100
    X_val, y_val = make_regression(n_samples=m, n_features=1, noise=5, random_state=42)

    # Sympy 심볼 정의
    w, b = sympy.symbols('w b')

    # 손실 함수 L(w, b) 정의: (1/2m) * sum((w*x + b - y)^2)
    loss_simple = 0
    for i in range(m):
        prediction = w * X_val[i][0] + b
        loss_simple += (prediction - y_val[i])**2
    loss_simple = (1 / (2 * m)) * loss_simple

    # 편미분 (Gradients)
    grad_w = sympy.diff(loss_simple, w)
    grad_b = sympy.diff(loss_simple, b)

    # 방정식 풀기: grad_w = 0, grad_b = 0
    sol_simple = sympy.solve([grad_w, grad_b], [w, b])

    print(f"도출된 가중치(w): {sol_simple[w]:.4f}")
    print(f"도출된 편향(b): {sol_simple[b]:.4f}")
    print("-" * 40)

    # ---------------------------------------------------------
    # 2. 다중 선형 회귀 (Multiple Linear Regression)
    # ---------------------------------------------------------
    print("\n--- [2] 다중 선형 회귀 (Symbolic) ---")

    # 실제 데이터 생성 (m=100 샘플, 특성 2개)
    X_multi, y_multi = make_regression(n_samples=m, n_features=2, noise=5, random_state=42)

    # Sympy 심볼 정의 (w1, w2, b)
    w1, w2, b_m = sympy.symbols('w1 w2 b_m')

    # 손실 함수 정의
    loss_multi = 0
    for i in range(m):
        # y_hat = w1*x1 + w2*x2 + b
        prediction = w1 * X_multi[i][0] + w2 * X_multi[i][1] + b_m
        loss_multi += (prediction - y_multi[i])**2
    loss_multi = (1 / (2 * m)) * loss_multi

    # 편미분
    dw1 = sympy.diff(loss_multi, w1)
    dw2 = sympy.diff(loss_multi, w2)
    db = sympy.diff(loss_multi, b_m)

    # 연립 방정식 풀기
    sol_multi = sympy.solve([dw1, dw2, db], [w1, w2, b_m])

    print(f"도출된 가중치 w1: {sol_multi[w1]:.4f}")
    print(f"도출된 가중치 w2: {sol_multi[w2]:.4f}")
    print(f"도출된 편향 b: {sol_multi[b_m]:.4f}")

if __name__ == "__main__":
    solve_regression_symbolic()

--- [1] 단순 선형 회귀 (Symbolic) ---
도출된 가중치(w): 43.0891
도출된 편향(b): 0.5826
----------------------------------------

--- [2] 다중 선형 회귀 (Symbolic) ---
도출된 가중치 w1: 86.8699
도출된 가중치 w2: 74.0968
도출된 편향 b: 0.1082


## 2. sklearn.linear_model_LinearRegression() 으로 구하기

In [3]:
from sklearn.linear_model import LinearRegression
import numpy as np

m = 100
X_val, y_val = make_regression(n_samples=m, n_features=1, noise=5, random_state=42)
X_multi, y_multi = make_regression(n_samples=m, n_features=2, noise=5, random_state=42)

In [4]:
import sklearn.linear_model
model = sklearn.linear_model.LinearRegression()
model.fit(X_val, y_val)
model.coef_

array([43.08913515])

In [5]:
sklearn.linear_model.LinearRegression().fit(X_val, y_val).coef_

array([43.08913515])

In [6]:
# 2. 단순회귀 모델 생성 및 학습
model_simple = LinearRegression()
model_simple.fit(X_val, y_val)

# 3. 결과 출력
print("--- 단순 선형 회귀 결과 ---")
print(f"모델이 예측한 가중치(w): {model_simple.coef_[0]:.2f}")
print(f"모델이 예측한 편향(b): {model_simple.intercept_:.2f}")

--- 단순 선형 회귀 결과 ---
모델이 예측한 가중치(w): 43.09
모델이 예측한 편향(b): 0.58


In [ ]:
# 2. 다중회귀모델 생성 및 학습
model_simple = LinearRegression()
model_simple.fit(X_multi, y_multi)

# 3. 결과 출력
print("--- 단순 선형 회귀 결과 ---")
print(f"모델이 예측한 가중치(w): {model_simple.coef_}")
print(f"모델이 예측한 편향(b): {model_simple.intercept_:.2f}")

--- 단순 선형 회귀 결과 ---
모델이 예측한 가중치(w): [86.86994374 74.09680794]
모델이 예측한 편향(b): 0.11


## 3. Keras로 구하기


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.datasets import make_regression

# 데이터 생성
X_simple, y_simple = make_regression(n_samples=100, n_features=1, noise=5, random_state=42)
X_multi, y_multi = make_regression(n_samples=100, n_features=3, noise=5, random_state=42)

def build_and_run_keras(X, y, title):
    model = models.Sequential([
        layers.Input(shape=(X.shape[1],)),  # 입력 특성 수 설정
        layers.Dense(1)                     # 출력층 (활성화 함수 없음 = 선형 회귀)
    ])

    model.compile(optimizer='sgd', loss='mse') # 확률적 경사 하강법 & 평균 제곱 오차
    model.fit(X, y, epochs=100, verbose=0)

    w, b = model.get_weights()
    print(f"[{title}] w: {w.flatten()}, b: {b}")

build_and_run_keras(X_simple, y_simple, "Keras 단순 회귀")
build_and_run_keras(X_multi, y_multi, "Keras 다중 회귀")

[Keras 단순 회귀] w: [43.06462], b: [0.7121528]
[Keras 다중 회귀] w: [27.995684 74.56402  18.21217 ], b: [0.5612033]


## 4. PyTorch (Object-Oriented API)
PyTorch는 모델 클래스를 직접 정의하거나 nn.Linear를 사용하여 구현합니다. 데이터 타입을 torch.Tensor로 변환하는 과정이 필요

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

# 데이터 준비 (Numpy -> Tensor)
X_s_torch = torch.FloatTensor(X_simple)
y_s_torch = torch.FloatTensor(y_simple).view(-1, 1)
X_m_torch = torch.FloatTensor(X_multi)
y_m_torch = torch.FloatTensor(y_multi).view(-1, 1)

def run_pytorch(X, y, title):
    input_dim = X.shape[1]
    model = nn.Linear(input_dim, 1) # y = wX + b 자동 생성

    criterion = nn.MSELoss()        # 앞서 배운 손실 함수 수식
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(500):
        optimizer.zero_grad()       # 기울기 초기화
        output = model(X)           # 예측 (Forward)
        loss = criterion(output, y) # 손실 계산
        loss.backward()             # 미분 (Backward - 기호 미분과 같은 원리)
        optimizer.step()            # 파라미터 업데이트

    print(f"[{title}] w: {model.weight.detach().numpy().flatten()}, b: {model.bias.item():.4f}")

run_pytorch(X_s_torch, y_s_torch, "PyTorch 단순 회귀")
run_pytorch(X_m_torch, y_m_torch, "PyTorch 다중 회귀")

[PyTorch 단순 회귀] w: [43.075184], b: 0.5765
[PyTorch 다중 회귀] w: [28.18424 74.50723 18.25446], b: 0.6372


- Scikit-learn: 수학적으로 완벽한 정답인 **정규 방정식(Normal Equation)** 을 사용하여 한 번에 해를 구합니다.

- Keras / PyTorch: **경사 하강법(Gradient Descent)** 이라는 반복적인 근사 계산법을 사용합니다. 학습 횟수(Epochs)나 학습률(Learning Rate)에 따라 정답에 가까워질 뿐, 완전히 똑같은 수치에 도달하지는 않습니다.

In [9]:
from sklearn.metrics import mean_squared_error

# 1. 데이터 생성 (다중 회귀: 특성 3개)
X, y = make_regression(n_samples=200, n_features=3, noise=10, random_state=42)

# 결과 저장용 딕셔너리
mse_results = {}

# ---------------------------------------------------------
# [Method 1] Scikit-learn (정밀한 수학적 해)
# ---------------------------------------------------------
skl_model = LinearRegression()
skl_model.fit(X, y)
y_pred_skl = skl_model.predict(X)
mse_results['Scikit-learn'] = mean_squared_error(y, y_pred_skl)

# ---------------------------------------------------------
# [Method 2] Keras / TensorFlow (반복적 최적화)
# ---------------------------------------------------------
tf_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),
    tf.keras.layers.Dense(1)
])
tf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.1), loss='mse')
tf_model.fit(X, y, epochs=200, verbose=0) # 학습
y_pred_tf = tf_model.predict(X).flatten()
mse_results['Keras/TF'] = mean_squared_error(y, y_pred_tf)

# ---------------------------------------------------------
# [Method 3] PyTorch (반복적 최적화)
# ---------------------------------------------------------
X_torch = torch.FloatTensor(X)
y_torch = torch.FloatTensor(y).view(-1, 1)
pt_model = torch.nn.Linear(3, 1)
optimizer = torch.optim.Adam(pt_model.parameters(), lr=0.1)
criterion = torch.nn.MSELoss()

for epoch in range(200):
    optimizer.zero_grad()
    outputs = pt_model(X_torch)
    loss = criterion(outputs, y_torch)
    loss.backward()
    optimizer.step()

y_pred_pt = pt_model(X_torch).detach().numpy().flatten()
mse_results['PyTorch'] = mean_squared_error(y, y_pred_pt)

# ---------------------------------------------------------
# 결과 출력
# ---------------------------------------------------------
print("\n--- 각 프레임워크별 최종 MSE 비교 ---")
for name, mse in mse_results.items():
    print(f"{name:15}: MSE = {mse:.6f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

--- 각 프레임워크별 최종 MSE 비교 ---
Scikit-learn   : MSE = 108.572329
Keras/TF       : MSE = 125.205863
PyTorch        : MSE = 5482.376659


### 왜 이렇게 차이가 있지? 그 이유는? 토론하라.